<a href="https://colab.research.google.com/github/TheAdnanP/BlogProject_foodcart/blob/main/ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot with TensorFlow and Keras

This repository contains a simple chatbot built using TensorFlow and Keras. The chatbot is designed to respond to common user queries based on predefined intents and patterns, utilizing text tokenization, sequence padding, and a neural network for intent classification. Key components include `intents.json` for conversational data, a `Tokenizer` for text processing, a `LabelEncoder` for intent labels, and a trained `chat_model.h5` for predictions.

In [ ]:
data = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["Hi", "Hey", "Is anyone there?", "Hello", "Hay"],
            "responses": ["Hello", "Hi", "Hi there"]
        },
        {
            "tag": "goodbye",
            "patterns": ["Bye", "See you later", "Goodbye"],
            "responses": ["See you later", "Have a nice day", "Bye! Come back again"]
        },
        {
            "tag": "thanks",
            "patterns": ["Thanks", "Thank you", "That's helpful", "Thanks for the help"],
            "responses": ["Happy to help!", "Any time!", "My pleasure", "You're most welcome!"]
        },
        {
            "tag": "about",
            "patterns": ["Who are you?", "What are you?", "Who you are?"],
            "responses": ["I'm Joana, your bot assistant", "I'm Joana, an Artificial Intelligent bot"]
        },
        {
            "tag": "name",
            "patterns": ["what is your name", "what should I call you", "whats your name?"],
            "responses": ["You can call me Joana.", "I'm Joana!", "Just call me Joana"]
        },
        {
            "tag": "help",
            "patterns": ["Could you help me?", "give me a hand please", "Can you help?", "What can you do for me?", "I need support", "I need help", "Support me please"],
            "responses": ["Tell me how I can assist you", "Tell me your problem so I can assist you", "Yes, sure, how can I support you?"]
        },
        {
            "tag": "createaccount",
            "patterns": ["I need to create a new account", "How to open a new account", "I want to create an account", "Can you create an account for me?", "How to open a new account"],
            "responses": ["You can easily create a new account from our website", "Just go to our website and follow the guidelines to create a new account"]
        },
        {
            "tag": "complaint",
            "patterns": ["I have a complaint", "I want to raise a complaint", "There is a complaint about a service"],
            "responses": ["Please provide us your complaint so we can assist you", "Please mention your complaint, we will reach out to you, and sorry for any inconvenience caused"]
        }
    ]
}

import json
import numpy as np
import tensorflow as tf
from tensorflow import keras

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder
import pickle

# Save intents to JSON file
with open('intents.json', 'w') as outfile:
    json.dump(data, outfile, indent=4)
print("Intents saved to 'intents.json'.")

Intents saved to 'intents.json'.


In [ ]:
# Data Preparation
training_sentences = []
training_labels = []
labels = []
responses = []
for intent in data['intents']:
    for pattern in intent['patterns']:
        training_sentences.append(pattern)
        training_labels.append(intent['tag'])
    responses.append(intent['responses'])
    if intent['tag'] not in labels:
        labels.append(intent['tag'])
num_classes = len(labels)

# Encode labels

lbl_encoder = LabelEncoder()
lbl_encoder.fit(training_labels)
training_labels_encoded = lbl_encoder.transform(training_labels)

# Save the label encoder
with open('label_encoder.pickle','wb') as enc:
  pickle.dump(lbl_encoder,enc,protocol=pickle.HIGHEST_PROTOCOL)

print("LabelEncoder saved to 'label_encoder.pickle&'.")


# Tokenization and Padding
vocab_size = 1000
embedding_dim = 16
max_len = 20
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size,
oov_token=oov_token)
tokenizer.fit_on_texts(training_sentences)
word_index = tokenizer.word_index
sequences = tokenizer.texts_to_sequences(training_sentences)
padded_sequences = pad_sequences(sequences,
truncating='post', maxlen=max_len)

# Save the tokenizer
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle,
protocol=pickle.HIGHEST_PROTOCOL)
print("Tokenizer saved to 'tokenizer.pickle'.")

LabelEncoder saved to 'label_encoder.pickle&'.
Tokenizer saved to 'tokenizer.pickle'.


In [ ]:
# Model Building
model = Sequential()
model.add(Embedding(vocab_size, embedding_dim,input_length=max_len))
model.add(GlobalAveragePooling1D())
model.add(Dense(16, activation= 'relu'))
model.add(Dense(16, activation= 'relu'))
model.add(Dense(num_classes, activation='softmax'))

# Compile the model
model.compile(loss='sparse_categorical_crossentropy',optimizer='adam', metrics=['accuracy'])

#MODEL SUMMARY
model.summary()

#Training the model
epochs = 500 # Consider reducing this number to
# prevent overfitting
history = model.fit(padded_sequences,
np.array(training_labels_encoded), epochs=epochs)

# Save the trained model
model.save("chat_model.h5")
print("Model saved to 'chat_model.h5'.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.0606 - loss: 2.0814
Epoch 2/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0909 - loss: 2.0789
Epoch 3/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.0909 - loss: 2.0785
Epoch 4/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0909 - loss: 2.0783
Epoch 5/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.0909 - loss: 2.0776
Epoch 6/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.0909 - loss: 2.0772
Epoch 7/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1818 - loss: 2.0764
Epoch 8/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1818 - loss: 2.0755
Epoch 9/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.1515 - loss: 2.0752
Epoch 10/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1515 - loss: 2.0746
Epoch 11/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.1515 - loss: 2.0741
Epoch 12/500
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.1515 - lo

Model saved to 'chat_model.h5'.


In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
import random
import pickle

# Load intents
with open("intents.json") as file:
    data = json.load(file)
# Load the trained model
model = keras.models.load_model('chat_model.h5')
print("Model loaded.")
# Load the tokenizer
with open('tokenizer.pickle', 'rb') as handle:
    tokenizer = pickle.load(handle)
print("Tokenizer loaded.")
# Load the label encoder
with open('label_encoder.pickle', 'rb') as enc:
    lbl_encoder = pickle.load(enc)
print("LabelEncoder loaded.")
def chat():
    max_len = 20 # Must be same as used during training
    while True:
        print("User: ", end="")
        inp = input()
        if inp.lower() == "quit":
            print("ChatBot: Goodbye! Have a great day!")
            break
        # Preprocess the input
        sequences = tokenizer.texts_to_sequences([inp])

        padded = keras.preprocessing.sequence.pad_sequences(sequences,
        truncating='post', maxlen=max_len)
        # Predict the intent
        result = model.predict(padded)
        predicted_label = lbl_encoder.inverse_transform([np.argmax(result)])[0]
        # Find the corresponding intent
        for intent in data['intents']:
            if intent['tag'] == predicted_label:
                response = random.choice(intent['responses'])
                print("ChatBot:", response)
                break
        else:
            # Fallback response if no intent is matched
            print("ChatBot: I'm sorry, I didn't understand that.\nCould you please rephrase?")
print("Start messaging with the bot (type 'quit' to stop)!")
chat()

Model loaded.
Tokenizer loaded.
LabelEncoder loaded.
Start messaging with the bot (type 'quit' to stop)!
User: hi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step
ChatBot: Hello
User: hey broh
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
ChatBot: Hi
User: what is 1+1
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
ChatBot: Hello
User: hay
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
ChatBot: Hi there
User: 

## Resources Used in this Project

**Python Libraries:**
* `json`: For handling JSON data (`intents.json`).
* `numpy`: For numerical operations, especially with arrays (`np.array`, `np.argmax`).
* `tensorflow` (`keras`): For building, compiling, and training the neural network model.
  * `tensorflow.keras.models.Sequential`: To create a sequential model.
  * `tensorflow.keras.layers.Dense`, `Embedding`, `GlobalAveragePooling1D`: For defining the layers of the neural network.
  * `tensorflow.keras.preprocessing.text.Tokenizer`: For text tokenization.
  * `tensorflow.keras.preprocessing.sequence.pad_sequences`: For padding sequences.
* `sklearn.preprocessing.LabelEncoder`: For encoding categorical labels into numerical format.
* `random`: For selecting random responses from the defined intents.
* `pickle`: For serializing and deserializing Python objects (like `LabelEncoder` and `Tokenizer`).

**Files Generated and Used:**
* `intents.json`: Stores the chatbot's conversational patterns and responses.
* `label_encoder.pickle`: Saves the `LabelEncoder` object, used to transform and inverse-transform labels.
* `tokenizer.pickle`: Saves the `Tokenizer` object, used to convert text into sequences for the model.
* `chat_model.h5`: The trained Keras model, saved in H5 format.